# 01.2 · Pipeline 模式（队列 + Worker 线程）

> 用消息队列解耦各 Agent，实现流水线并行。


In [4]:
import nest_asyncio; nest_asyncio.apply()
import queue, threading, asyncio, time, uuid, random, copy
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

# ── 模拟 LLM / 向量检索 ─────────────────────────────────────────────────────
CORPUS = [
    "RAG 通过检索外部文档来扩充 LLM 的上下文。",
    "向量数据库支持语义相似度搜索。",
    "Reranker 用交叉编码器对候选文档重新打分。",
    "幂等设计让系统在重试时不产生副作用。",
    "Backpressure 防止下游服务因流量过大而崩溃。",
    "可观测性包括 Metrics、Traces 和 Logs 三大支柱。",
    "Graceful degradation 让系统在部分故障时仍能返回有用结果。",
    "Hub-and-Spoke 模式由 Orchestrator 统一分派任务。",
    "Blackboard 模式通过共享工作区实现多 Agent 协作。",
    "消息队列解耦生产者和消费者，支持异步处理。",
]

def fake_retrieve(query: str, top_k: int = 5, latency: float = 0.3) -> list[dict]:
    """模拟向量检索，返回 top_k 相关文档"""
    time.sleep(latency)
    scored = [(doc, random.uniform(0.5, 1.0)) for doc in CORPUS]
    scored.sort(key=lambda x: -x[1])
    return [{"text": doc, "score": round(s, 3)} for doc, s in scored[:top_k]]

def fake_rerank(query: str, hits: list[dict], top_k: int = 3, latency: float = 0.2) -> list[dict]:
    """模拟 cross-encoder reranker"""
    time.sleep(latency)
    reranked = copy.deepcopy(hits)
    for h in reranked:
        h["score"] = round(h["score"] * random.uniform(0.9, 1.1), 3)
    reranked.sort(key=lambda x: -x["score"])
    return reranked[:top_k]

def fake_generate(question: str, context: list[dict], latency: float = 0.5) -> str:
    """模拟 LLM 生成"""
    time.sleep(latency)
    snippets = " ".join(h["text"][:30] for h in context[:2])
    return f"[答案] 关于「{question[:20]}」：{snippets}..."

def fake_verify(answer: str, context: list[dict], latency: float = 0.15) -> dict:
    """模拟答案校验"""
    time.sleep(latency)
    return {"grounded": random.random() > 0.2, "confidence": round(random.uniform(0.7, 1.0), 2)}

print("✅ 模拟工具加载完毕")

✅ 模拟工具加载完毕


---
## Part 2 · Pipeline 模式（队列 + Worker 线程）

用 Python 内置 `queue.Queue` 模拟 Redis 列表，每个 Worker 是一个线程。

```
Orchestrator
     │ put(job)
     ▼
q_retrieve ──▶ RetrieverWorker
                    │ put(job+hits)
                    ▼
              q_rerank ──▶ RerankerWorker
                                │ put(job+topk)
                                ▼
                          q_generate ──▶ GeneratorWorker
                                              │ put(result)
                                              ▼
                                         results[job_id]
```

In [5]:
# ── 队列定义 ─────────────────────────────────────────────────────────────────
q_retrieve = queue.Queue()
q_rerank   = queue.Queue()
q_generate = queue.Queue()
results    = {}          # job_id → result（生产中用 Redis / DB）
_stop      = threading.Event()

# ── 流水线事件日志（线程安全，由 Cell 4 统一 drain 打印）────────────────────
_log_lock = threading.Lock()
_pipeline_log: list[str] = []
_t0 = time.perf_counter()

def _elapsed() -> str:
    return f"{time.perf_counter() - _t0:6.3f}s"

def log_event(stage: str, job_id: str, msg: str):
    line = f"  {_elapsed()} │ [{stage:>9}] {job_id[:8]} │ {msg}"
    with _log_lock:
        _pipeline_log.append(line)

def drain_pipeline_log(start: int = 0) -> int:
    """打印自 start 索引以来的新日志，返回新索引。"""
    with _log_lock:
        new_lines = _pipeline_log[start:]
        end = len(_pipeline_log)
    for line in new_lines:
        print(line, flush=True)
    return end

def reset_pipeline_log():
    global _t0
    with _log_lock:
        _pipeline_log.clear()
    _t0 = time.perf_counter()

# ── Worker 定义 ───────────────────────────────────────────────────────────────
def retriever_worker():
    while not _stop.is_set():
        try:
            job = q_retrieve.get(timeout=0.2)
        except queue.Empty:
            continue
        rid, q = job["request_id"], job["query"]
        log_event("Retriever", rid, f"▶ 开始 │ query={q[:18]!r} │ q_retrieve={q_retrieve.qsize()}")
        t0 = time.perf_counter()
        hits = fake_retrieve(job["query"], latency=0.3)
        job = {**job, "hits": hits}
        q_rerank.put(job)
        log_event("Retriever", rid,
                  f"✓ 完成 {time.perf_counter()-t0:.2f}s │ hits={len(hits)} "
                  f"top_score={hits[0]['score']} → q_rerank({q_rerank.qsize()})")
        q_retrieve.task_done()

def reranker_worker():
    while not _stop.is_set():
        try:
            job = q_rerank.get(timeout=0.2)
        except queue.Empty:
            continue
        rid, q = job["request_id"], job["query"]
        log_event("Reranker", rid,
                  f"▶ 开始 │ query={q[:18]!r} │ in_hits={len(job['hits'])} │ q_rerank={q_rerank.qsize()}")
        t0 = time.perf_counter()
        topk = fake_rerank(job["query"], job["hits"], latency=0.2)
        job = {**job, "topk": topk}
        q_generate.put(job)
        scores = ", ".join(str(h["score"]) for h in topk)
        log_event("Reranker", rid,
                  f"✓ 完成 {time.perf_counter()-t0:.2f}s │ query={q[:18]!r} │ "
                  f"topk={len(topk)} scores=[{scores}] → q_generate({q_generate.qsize()})")
        q_rerank.task_done()

def generator_worker():
    while not _stop.is_set():
        try:
            job = q_generate.get(timeout=0.2)
        except queue.Empty:
            continue
        rid, q = job["request_id"], job["query"]
        log_event("Generator", rid,
                  f"▶ 开始 │ query={q[:18]!r} │ ctx={len(job['topk'])} docs │ q_generate={q_generate.qsize()}")
        t0 = time.perf_counter()
        answer  = fake_generate(job["query"], job["topk"], latency=0.5)
        verdict = fake_verify(answer, job["topk"], latency=0.15)
        results[job["request_id"]] = {
            "answer": answer,
            "verdict": verdict,
            "finished_at": time.perf_counter(),
        }
        log_event("Generator", rid,
                  f"✓ 完成 {time.perf_counter()-t0:.2f}s │ query={q[:18]!r} │ "
                  f"grounded={verdict['grounded']} conf={verdict['confidence']} │ answer={answer[:40]}...")
        q_generate.task_done()

# ── 启动 / 停止 Workers（可重复调用）──────────────────────────────────────────
workers: list[threading.Thread] = []

def start_workers():
    """启动 Worker；若上次被 stop 了，会自动重建线程。"""
    global workers
    if any(w.is_alive() for w in workers):
        return
    _stop.clear()
    workers = [
        threading.Thread(target=retriever_worker, daemon=True),
        threading.Thread(target=reranker_worker,  daemon=True),
        threading.Thread(target=generator_worker, daemon=True),
    ]
    for w in workers:
        w.start()

def stop_workers():
    """通知 Worker 退出（下次 start_workers() 会重建）。"""
    _stop.set()

start_workers()
print("✅ 3 个 Worker 已启动（Retriever / Reranker / Generator）")

✅ 3 个 Worker 已启动（Retriever / Reranker / Generator）


In [6]:
# ── 提交多个查询，观察并发流水线 ────────────────────────────────────────────
start_workers()
reset_pipeline_log()

queries = ["什么是 RAG？", "幂等设计是什么？", "如何实现可观测性？"]
job_ids = []
t_start = time.perf_counter()

print("📤 提交阶段")
for q in queries:
    job_id = str(uuid.uuid4())
    job_ids.append((job_id, q, time.perf_counter()))
    q_retrieve.put({"request_id": job_id, "query": q})
    print(f"  ➡ {q[:20]:<20} [{job_id[:8]}] → q_retrieve({q_retrieve.qsize()})", flush=True)

# 等待所有结果，同时实时打印流水线事件
print("\n📊 流水线事件（时间 │ 阶段 │ job_id │ 详情）")
print("  " + "─" * 72)
_log_cursor = 0
pending = {job_id: (query, t_submit) for job_id, query, t_submit in job_ids}
TIMEOUT = 30
deadline = time.perf_counter() + TIMEOUT

while pending:
    _log_cursor = drain_pipeline_log(_log_cursor)
    for job_id in list(pending):
        if job_id not in results:
            continue
        query, t_submit = pending.pop(job_id)
        r = results[job_id]
        latency = round(r["finished_at"] - t_submit, 3)
        print(f"  🏁 [{job_id[:8]}] 完成 │ {query[:20]} │ {latency}s │ grounded={r['verdict']['grounded']}", flush=True)
    if time.perf_counter() > deadline:
        raise TimeoutError(
            f"仍有 {len(pending)} 个任务未完成：{[k[:8] for k in pending]}。"
            "请先重跑上一个 cell，或确认 start_workers() 已执行。"
        )
    time.sleep(0.05)

_log_cursor = drain_pipeline_log(_log_cursor)
print("  " + "─" * 72)

total = round(time.perf_counter() - t_start, 3)
print(f"\n📈 汇总")
print(f"  总耗时：{total}s（3 个查询并行流水线，理想 ≈ 单次 ~1.2s，而非 3×）")
print(f"  队列清空：retrieve={q_retrieve.qsize()} rerank={q_rerank.qsize()} generate={q_generate.qsize()}")

stop_workers()

📤 提交阶段
  ➡ 什么是 RAG？             [68313364] → q_retrieve(1)
  ➡ 幂等设计是什么？             [ae6a5532] → q_retrieve(1)
  ➡ 如何实现可观测性？            [48c5e5e7] → q_retrieve(2)

📊 流水线事件（时间 │ 阶段 │ job_id │ 详情）
  ────────────────────────────────────────────────────────────────────────
   0.001s │ [Retriever] 68313364 │ ▶ 开始 │ query='什么是 RAG？' │ q_retrieve=0
   0.302s │ [Retriever] 68313364 │ ✓ 完成 0.30s │ hits=5 top_score=0.955 → q_rerank(1)
   0.302s │ [Retriever] ae6a5532 │ ▶ 开始 │ query='幂等设计是什么？' │ q_retrieve=1
   0.302s │ [ Reranker] 68313364 │ ▶ 开始 │ query='什么是 RAG？' │ in_hits=5 │ q_rerank=0
   0.502s │ [ Reranker] 68313364 │ ✓ 完成 0.20s │ query='什么是 RAG？' │ topk=3 scores=[0.967, 0.887, 0.855] → q_generate(1)
   0.502s │ [Generator] 68313364 │ ▶ 开始 │ query='什么是 RAG？' │ ctx=3 docs │ q_generate=0
   0.602s │ [Retriever] ae6a5532 │ ✓ 完成 0.30s │ hits=5 top_score=0.979 → q_rerank(1)
   0.602s │ [Retriever] 48c5e5e7 │ ▶ 开始 │ query='如何实现可观测性？' │ q_retrieve=0
   0.602s │ [ Reranker] ae6a5532 │ ▶ 开始 │ query

> **关键观察**：
> - 三个查询**同时**在流水线中流动，总时间 ≈ 单次时间（而非 3×）
> - 每个 Worker 职责单一，替换 Reranker 不影响其他两个
> - 队列天然起到**背压缓冲**作用

## 📖 讲义

# Part 2
## 核心概念：设计原则与架构模式

---

# 多 Agent 的四大设计原则

<div class="columns">
<div>

### 1. 职责分离 SRP
每个 Agent 只做一件明确的事
- Retriever → 只负责检索
- Reranker  → 只负责重排
- Generator → 只负责生成
- Verifier  → 只负责校验

</div>
<div>

### 2. 松耦合通信
- 通过消息/队列/API 交换数据
- 不直接调用对方内部函数
- 接口契约（schema）稳定

### 3. 可独立扩展
- 每个 Agent 独立部署
- 按负载单独扩缩容

### 4. 可替换性
- 换掉一个 Agent 不影响其他

</div>
</div>

---

# 架构模式 1：Pipeline（流水线）

```
请求
 │
 ▼
[Retriever] ──▶ [Reranker] ──▶ [Generator] ──▶ [Verifier] ──▶ 响应
```

**特点**
- 线性、顺序处理，易于理解与调试
- 每个阶段输出即下一阶段输入
- 适合：文档 QA、内容审核、数据清洗流水线

**实现方式**
- 消息队列（Redis / Kafka）：每个 Worker 消费上游队列，写入下游队列
- 同步链式调用：Orchestrator 依次调用各服务

---

# 架构模式 2：Hub-and-Spoke（协调器）

```
                ┌─── Retriever ──┐
                │                │
用户 ──▶ Orchestrator ─── Reranker ───▶ 合成回复
                │                │
                └─── Generator ──┘
```

**特点**
- Orchestrator 负责任务分派与结果合成
- 支持并行调用多个 Agent
- 适合：复杂问答、多步推理、工作流编排

**注意**
- Orchestrator 成为单点，需做好容错与限流
- 建议用异步 + 回调，避免阻塞

---

# 架构模式 3：Blackboard（共享工作区）

```
          写入              读取
Agent A ──────▶ Blackboard ◀────── Agent C
Agent B ──────▶     │     ◀────── Agent D
                    │
              变更通知（事件）
```

**特点**
- 多个 Agent 共享同一"黑板"（数据库/缓存/对象存储）
- 通过事件/轮询感知变化，异步推进
- 适合：迭代协作、多模型协商、长任务

**挑战**
- 需要设计好写入合约与冲突处理
- 数据清理策略（TTL / GC）

---

# 三种模式对比

| 维度 | Pipeline | Hub-and-Spoke | Blackboard |
|------|----------|---------------|------------|
| 复杂度 | 低 | 中 | 高 |
| 调试难度 | 易 | 中 | 难 |
| 并行能力 | 低 | 高 | 高 |
| 适合任务 | 线性 | 多角色协作 | 迭代协商 |
| 典型实现 | Redis Queue | FastAPI + gRPC | DB + Pub-Sub |

> **建议**：从 Pipeline 起步，按需演进到 Hub-and-Spoke，Blackboard 用于高级场景

---

# 协调策略

<div class="columns">
<div>

### 中心化编排
```
Orchestrator
 ├─ 分配任务
 ├─ 控制顺序
 └─ 合成结果
```
✅ 可控、易监控、易审计
❌ 单点复杂性、扩展受限

</div>
<div>

### 去中心化 / 涌现式
```
Agent A ◀──▶ Agent B
  ▲               ▲
  └───▶ Agent C ──┘
```
✅ 高鲁棒性、天然分布式
❌ 设计复杂、调试困难

### Hybrid（推荐）
中心化编排 + 局部自治
平衡可控性与弹性

</div>
</div>

---

---
**导航**：[← 01.1 · 单体 Agent 的局限](./01.1_monolithic_agent.ipynb) · [📋 课程索引](./01_multi_agent_arch_demo.ipynb) · [01.3 · Hub-and-Spoke 模式 →](./01.3_hub_and_spoke.ipynb)
